# 05 — Clean Room Graph
Rebuild the building from structural corners, add apertures, draw adjacency + access graphs.

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
import colorsys

print(Helper.Version())
renderer = "vscode"
APTS = "c:/Users/etmaglari/IAAC/etmaglari_gML/Rhino_Apertures _etm.obj"

## 1. Room helper — explicit face construction, correct outward normals

In [ ]:
def make_room(xz_polygon, y0=0, y1=10):
    """
    Create a Cell by extruding a floor polygon (list of (X,Z) tuples) in Y.
    All face normals point outward so Cell.ByFaces builds a valid cell.
    """
    n = len(xz_polygon)
    bot = [Vertex.ByCoordinates(x, y0, z) for x, z in xz_polygon]
    top = [Vertex.ByCoordinates(x, y1, z) for x, z in xz_polygon]

    faces = []
    # Floor: original order → normal points down (outward)
    faces.append(Face.ByVertices(bot))
    # Ceiling: reversed order → normal points up (outward)
    faces.append(Face.ByVertices(list(reversed(top))))
    # Walls: [bot[i], top[i], top[j], bot[j]] → outward normal
    for i in range(n):
        j = (i + 1) % n
        faces.append(Face.ByVertices([bot[i], top[i], top[j], bot[j]]))

    return Cell.ByFaces(faces)


def box(x0, z0, x1, z1, y0=0, y1=10):
    """Shorthand for a rectangular room from floor-plan corners (X,Z)."""
    xmn, xmx = min(x0, x1), max(x0, x1)
    zmn, zmx = min(z0, z1), max(z0, z1)
    return make_room([(xmn, zmn), (xmx, zmn), (xmx, zmx), (xmn, zmx)], y0, y1)

## 2. Define the 10 rooms and build the CellComplex

In [ ]:
rooms = [
    # ── N-S wing ──────────────────────────────────────────────────────────────
    box(75, -7,  99, 17),                                                    # north room
    box(75, -43, 78, -7),                                                    # west corridor
    make_room([(78,-7),(99,-7),(99,-22),(86,-22),(86,-19),(78,-19)]),         # NE room (L-shape)
    box(78, -25, 86, -19),                                                   # inner room A
    box(78, -31, 86, -25),                                                   # inner room B
    make_room([(78,-31),(78,-43),(99,-43),(99,-22),(86,-22),(86,-31)]),       # south room (L-shape)
    # ── E-W wing ──────────────────────────────────────────────────────────────
    make_room([(35,-24),(60,-24),(60,-7),(75,-7),(75,0),(35,0)]),             # central room (L-shape)
    box(5, -17, 15, -7),                                                     # small room B
    make_room([(0,-17),(0,0),(35,0),(35,-24),(15,-24),(15,-7),(5,-7),(5,-17)]), # main room
    box(0, -24, 15, -17),                                                    # SW room
]

cc = CellComplex.ByCells(rooms)
cells = Topology.Cells(cc)
print(f"{len(cells)} cells in CellComplex")

In [ ]:
# Colour each room and show
n = len(cells)
colors = ["#%02x%02x%02x" % tuple(int(c*255) for c in colorsys.hsv_to_rgb(i/n, 0.6, 0.9))
          for i in range(n)]
for i, cell in enumerate(cells):
    Topology.SetDictionary(cell, Dictionary.ByKeysValues(["color","id"], [colors[i], i]))

Topology.Show(cc,
              faceColorKey="color", faceOpacity=0.5,
              edgeColor="white", edgeWidth=1,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 3. Load apertures

In [ ]:
apt_objects = Topology.ByOBJPath(APTS)
apertures   = Topology.Faces(Cluster.ByTopologies(apt_objects))
print(f"{len(apertures)} apertures loaded")

cc_apts = Topology.AddApertures(cc, apertures)
print("Done")

## 4. Adjacency graph — every shared wall = edge

In [ ]:
g_adj = Graph.ByTopology(cc_apts)
print(f"Adjacency: {len(Graph.Vertices(g_adj))} nodes, {len(Graph.Edges(g_adj))} edges")

for v in Graph.Vertices(g_adj):
    Topology.SetDictionary(v, Dictionary.ByKeysValues(["size","color"], [15, "red"]))
for e in Graph.Edges(g_adj):
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width","color"], [2, "white"]))

Topology.Show(cc_apts, g_adj,
              faceColor="white", faceOpacity=0.05,
              edgeColor="white", edgeWidth=1,
              vertexSizeKey="size", vertexColorKey="color",
              edgeWidthKey="width", edgeColorKey="color",
              showVertices=True,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 5. Access graph — only aperture-connected rooms = edge

In [ ]:
g_access = Graph.ByTopology(cc_apts, direct=False, viaSharedApertures=True)
print(f"Access: {len(Graph.Vertices(g_access))} nodes, {len(Graph.Edges(g_access))} edges")

for v in Graph.Vertices(g_access):
    Topology.SetDictionary(v, Dictionary.ByKeysValues(["size","color"], [15, "red"]))
for e in Graph.Edges(g_access):
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width","color"], [3, "#2CFF96"]))

Topology.Show(cc_apts, g_access,
              faceColor="white", faceOpacity=0.05,
              edgeColor="white", edgeWidth=1,
              vertexSizeKey="size", vertexColorKey="color",
              edgeWidthKey="width", edgeColorKey="color",
              showVertices=True,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)